"""Default script for making predictions with an ALSmodel and saving them to disk.

NOTE: performance is quite slow for brand data (~2h) given data size

"""

In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
from dateutil import parser
from math import ceil
import os
import sys
sys.path.append('..')
sys.path.append('../..')


from pyspark import SparkContext
from pyspark import SparkConf
from pyspark import StorageLevel
from pyspark.ml.recommendation import ALSModel
from pyspark.sql import SparkSession

from lib_cf.cf_io import read_scaler

from lib_cf.busrules import (
    limit_based_on_prior_trips,
    limit_based_on_score,
)
from lib_cf.busrules import (
    apply_bus_rules,
    label_hook_stretch,
)
from lib_cf.models import apply_output_scaling
from lib_cf.models import predict_pf
from lib_cf.models import add_cf_reg
from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
)
from lib.utils import top_n

import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) ---- ARGUMENTS ---- #

CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(list(CNF["shared"].items()) + list(CNF["predict"].items()))
PATHS = CNF["paths"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)
PARAMS["pred_date"] = parser.parse(PARAMS["pred_date"])

if PARAMS["future_run"]:
    tag = 'future'
else:
    tag = 'champion'

cf_model_uri = f"models:/{cf_model_catalog}_{PARAMS['category'].lower()}@{tag}"

In [0]:
# (3) ---- READ DATA ---- #

print("(1/3) reading data...")
model = mlflow.spark.load_model(cf_model_uri)
slate = read_cf_tables(cf_slate, PARAMS)
pdata = read_cf_tables(cf_matrix, PARAMS, past_flag=True)
if PARAMS["future_run"]:
    fdata = read_cf_tables(cf_matrix, PARAMS, future_flag=True)
cat_lookup = read_cf_tables(cf_cat_lookup, PARAMS)
if PARAMS["test"]:
    cube = spark.read.csv(PATHS["CUBE"], header=True)
else:
    cube = spark.table(fs_customer_cube_full)

In [0]:
# (4) ---- PREDICT ---- #

print("(2/3) making predictions...")
if PARAMS["future_run"]:
    predictions = predict_pf(
        PARAMS,
        model,
        slate,
        pdata,
        fdata,
        cat_lookup=cat_lookup,
        col_of_interest="TRIPS",
    )
else:
    predictions = predict_pf(
        PARAMS,
        model,
        slate,
        pdata,
        cat_lookup=cat_lookup,
        col_of_interest="TRIPS",
    )
predictions = predictions.repartition("CATEGORY_ID")
predictions.cache()
print("recommending on ", predictions.count(), " raw predictions")

# scale if appropriate
if PARAMS["scale"]:
    scaler_path = PATHS["scaler"]
    scaler = read_scaler(sc, scaler_path)
    predictions = apply_output_scaling(predictions, "prediction", scaler)
# apply business rules to generate output scores
predictions = apply_bus_rules(
    spark,
    PATHS,
    predictions,
    cube,
    "prediction",
    PARAMS["pred_date"],
    PARAMS["category"],
    PARAMS["rel_weighting"],
)

# label hook-stretch
predictions = label_hook_stretch(spark, PARAMS, PATHS, predictions)

# add predictions with empirical insight
predictions = add_cf_reg(spark, PARAMS, predictions)

# subset, cutoffs, score limits, etc...
if PARAMS["stretch"]:
    predictions = predictions[predictions.hs_ind == "stretch"]
if PARAMS["hook"]:
    predictions = predictions[predictions.hs_ind == "hook"]
if PARAMS["subset"] is not None:
    print(PARAMS["subset"])
    predictions = predictions[predictions.MBRSHP_SID.isin(PARAMS["subset"])]
if PARAMS["cutoff"] is not None:
    predictions = limit_based_on_prior_trips(
        spark, PATHS, predictions, PARAMS["data"], PARAMS["cutoff"]
    )
if PARAMS["score_limit"] is not None:
    predictions = limit_based_on_score(
        predictions, "prediction", PARAMS["score_limit"]
    )

predictions = top_n(
    predictions, PARAMS["recommendations"], "prediction", "MBRSHP_SID"
)
if PARAMS["future_run"]:
    predictions = predictions.drop("PAST_TRIPS", "FUTURE_TRIPS")
else:
    predictions = predictions.drop("PAST_TRIPS")
predictions.cache()
print("made ", predictions.count(), "final predictions")

In [0]:
save_cf_tables(predictions, cf_prediction, PARAMS)